# 00 — Synthetic OTEL Trace Generation

**Purpose:** Fabricate a realistic dataset of **OTEL / MLflow GenAI traces** and write it to a Delta table, so the rest of this blueprint is runnable without a live trace store.

In production your traces already exist — every LLM call logged through MLflow Tracing (or an OpenTelemetry collector) produces a span with the request messages and the model's response. Here we synthesize that same shape for a small **customer-support SQL assistant** so downstream notebooks have a learnable signal.

---

## What one trace looks like

Each row models a single LLM-call span following the [OpenTelemetry GenAI semantic conventions](https://opentelemetry.io/docs/specs/semconv/gen-ai/) and MLflow's trace fields:

| Column | Meaning |
|---|---|
| `trace_id` / `span_id` | span identity |
| `timestamp_ms` | span start time |
| `status` | `OK` or `ERROR` |
| `attributes` | JSON string: `gen_ai.system`, `gen_ai.request.model`, `mlflow.spanInputs` (messages), `mlflow.spanOutputs` (reply) |

Storing `attributes` as a JSON string mirrors how trace payloads arrive from MLflow/OTEL, so `01` parses it the same way it would parse real data.

In [ ]:
# Trace generation needs no extra libraries beyond the Databricks runtime.
# (No %pip install required for this notebook.)

In [ ]:
# ─────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────
CATALOG      = "main"
SCHEMA       = "otel_finetuning"
TRACES_TABLE = f"{CATALOG}.{SCHEMA}.synthetic_traces"

N_TRACES     = 500        # total spans to generate
ERROR_RATE   = 0.08       # fraction of ERROR / empty-output spans (exercises 01's filter)
SEED         = 42         # deterministic generation

GEN_AI_SYSTEM = "databricks"
GEN_AI_MODEL  = "databricks-meta-llama-3-1-8b-instruct"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

## Domain content

A compact, coherent domain gives fine-tuning something to learn. This assistant answers business questions by returning a short explanation plus a SQL query against a fixed schema. We build a pool of `(user_question, assistant_answer)` templates and sample from it.

In [ ]:
import random, json, hashlib

SYSTEM_PROMPT = (
    "You are a helpful data assistant for an e-commerce company. "
    "Given a business question, respond with a one-sentence explanation "
    "followed by a single SQL query against the `sales`, `customers`, and "
    "`products` tables. Wrap SQL in a ```sql code block."
)

# (question, sql) templates — assistant answer = explanation + SQL block
TEMPLATES = [
    ("How many orders were placed last month?",
     "SELECT count(*) FROM sales WHERE order_date >= date_trunc('month', current_date) - interval 1 month AND order_date < date_trunc('month', current_date);"),
    ("What are the top 5 products by revenue?",
     "SELECT p.name, sum(s.amount) AS revenue FROM sales s JOIN products p ON s.product_id = p.id GROUP BY p.name ORDER BY revenue DESC LIMIT 5;"),
    ("Which customers spent more than $1000 in total?",
     "SELECT c.name, sum(s.amount) AS total FROM sales s JOIN customers c ON s.customer_id = c.id GROUP BY c.name HAVING total > 1000;"),
    ("What is the average order value by month?",
     "SELECT date_trunc('month', order_date) AS month, avg(amount) AS avg_order FROM sales GROUP BY 1 ORDER BY 1;"),
    ("How many new customers signed up this year?",
     "SELECT count(*) FROM customers WHERE year(signup_date) = year(current_date);"),
    ("What is the total revenue by product category?",
     "SELECT p.category, sum(s.amount) AS revenue FROM sales s JOIN products p ON s.product_id = p.id GROUP BY p.category ORDER BY revenue DESC;"),
    ("Which products have never been ordered?",
     "SELECT p.name FROM products p LEFT JOIN sales s ON p.id = s.product_id WHERE s.product_id IS NULL;"),
    ("What is the month-over-month revenue growth?",
     "SELECT month, revenue, revenue - lag(revenue) OVER (ORDER BY month) AS growth FROM (SELECT date_trunc('month', order_date) AS month, sum(amount) AS revenue FROM sales GROUP BY 1);"),
]

EXPLANATIONS = [
    "Here is a query that answers your question.",
    "You can compute this as follows.",
    "This aggregates the relevant rows.",
    "The following query returns what you need.",
]

def build_answer(rng, sql):
    return f"{rng.choice(EXPLANATIONS)}\n\n```sql\n{sql}\n```"

In [ ]:
def make_trace(i, rng):
    """Build a single OTEL/MLflow-style trace row."""
    q, sql = rng.choice(TEMPLATES)
    is_error = rng.random() < ERROR_RATE

    inputs = {"messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": q},
    ]}

    if is_error:
        status = "ERROR"
        outputs = None                      # failed call → no usable completion
    else:
        status = "OK"
        outputs = {"choices": [{"message": {"role": "assistant",
                                            "content": build_answer(rng, sql)}}]}

    attributes = {
        "gen_ai.system": GEN_AI_SYSTEM,
        "gen_ai.request.model": GEN_AI_MODEL,
        "gen_ai.request.temperature": 0.2,
        "mlflow.spanInputs": inputs,
        "mlflow.spanOutputs": outputs,
    }

    tid = hashlib.sha1(f"trace-{i}".encode()).hexdigest()[:32]
    sid = hashlib.sha1(f"span-{i}".encode()).hexdigest()[:16]
    return {
        "trace_id": tid,
        "span_id": sid,
        "timestamp_ms": 1_700_000_000_000 + i * 1000,
        "status": status,
        "attributes": json.dumps(attributes),   # JSON string, like real trace payloads
    }

rng = random.Random(SEED)
rows = [make_trace(i, rng) for i in range(N_TRACES)]
print(f"Generated {len(rows)} traces "
      f"({sum(r['status']=='ERROR' for r in rows)} ERROR).")
print(json.dumps(json.loads(rows[0]['attributes']), indent=2)[:600])

## Write to Delta

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, LongType

schema = StructType([
    StructField("trace_id", StringType()),
    StructField("span_id", StringType()),
    StructField("timestamp_ms", LongType()),
    StructField("status", StringType()),
    StructField("attributes", StringType()),
])

df = spark.createDataFrame(rows, schema=schema)
(df.write.mode("overwrite").saveAsTable(TRACES_TABLE))

print(f"Wrote {df.count()} rows to {TRACES_TABLE}")
display(spark.table(TRACES_TABLE).limit(5))